In [7]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import joblib
import torch
from pathlib import Path
from scipy.sparse import hstack, csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
import torch.nn as nn
import re

#Loading saved models 
CACHE_DIR = Path("./cache")
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
st_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

tfidf = joblib.load(CACHE_DIR / "tfidf_vectorizer.joblib")
label_enc = joblib.load(CACHE_DIR / "label_encoder.joblib")

log_reg  = joblib.load("lr_model.joblib")
xgb_model = joblib.load("xgb_model.joblib")
lgb_model = joblib.load("lgb_model.joblib")
cat_model = joblib.load("cat_model.joblib")

class ImprovedMLP(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.35),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.30),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.25),

            nn.Linear(64, output_dim)
        )

    def forward(self, x):
        return self.net(x)

INPUT_DIM = 6542
OUTPUT_DIM = len(label_enc.classes_)

mlp_model = ImprovedMLP(INPUT_DIM, OUTPUT_DIM).to(device)
mlp_model.load_state_dict(torch.load("mlp_model.pth", map_location=device))
mlp_model.eval()


#Specifying Flags and Functions
def clean_text(s):
    if s is None: return ""
    s = re.sub(r"<.*?>", " ", str(s))
    s = re.sub(r"http\S+|www\.\S+", " ", s)
    s = re.sub(r"[^a-zA-Z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip().lower()

def map_verifiable(x):
    s = str(x).lower()
    return 1 if "verif" in s and not s.startswith("non") and "not" not in s else 0

def extract_nums(s):
    return re.findall(r"\d+", str(s))

def num_contradiction(c, e):
    cnums, enums = set(extract_nums(c)), set(extract_nums(e))
    if not cnums or not enums: return -1
    return 1 if cnums.isdisjoint(enums) else 0

def semantic_conflict(c, e, threshold=0.35):
    emb_c = st_model.encode(c, convert_to_tensor=True)
    emb_e = st_model.encode(e, convert_to_tensor=True)
    sim = util.cos_sim(emb_c, emb_e).item()
    if sim < -0.15: return 2
    if sim < threshold: return 1
    return 0

def negation_mismatch(c, e):
    neg_words = {"not", "no", "never", "none", "doesn't", "isnt", "isn't"}
    c_neg = any(w in c.split() for w in neg_words)
    e_neg = any(w in e.split() for w in neg_words)
    return int(c_neg != e_neg)

def direct_negation_conflict(c, e):
    e_words = e.split()
    c_words = c.split()
    for i in range(len(e_words) - 1):
        if e_words[i] == "not" and e_words[i+1] in c_words:
            return 1
    return 0

def predict_claim(claim, evidence, verifiable="VERIFIABLE"):

    # Cleaning
    c_clean = clean_text(claim)
    e_clean = clean_text(evidence)
    verif_flag = map_verifiable(verifiable)

    # Embeddings
    c_emb = st_model.encode(c_clean, convert_to_numpy=True)
    e_emb = st_model.encode(e_clean, convert_to_numpy=True)

    # Handcrafted features
    num_contra = num_contradiction(c_clean, e_clean)
    sem_conf = semantic_conflict(c_clean, e_clean)
    cos_sim = cosine_similarity(c_emb.reshape(1,-1), e_emb.reshape(1,-1))[0,0]
    neg_flag = negation_mismatch(c_clean, e_clean)
    directneg = direct_negation_conflict(c_clean, e_clean)
    handcrafted = np.array([[verif_flag, num_contra, sem_conf, neg_flag, directneg]])

    # Combine all embedding features
    emb_features = np.hstack([
        np.abs(c_emb - e_emb).reshape(1, -1),
        (c_emb * e_emb).reshape(1, -1),
        c_emb.reshape(1, -1),
        e_emb.reshape(1, -1),
        np.array([[cos_sim]]),
        handcrafted
    ])

    X_final = hstack([csr_matrix(emb_features), tfidf.transform([c_clean + " " + e_clean])])

    # Model predictions
    lr_pred = label_enc.inverse_transform([log_reg.predict(X_final)[0]])[0]
    xgb_pred = label_enc.inverse_transform([xgb_model.predict(X_final)[0]])[0]
    lgb_pred = label_enc.inverse_transform([lgb_model.predict(X_final)[0]])[0]
    cat_pred = label_enc.inverse_transform([int(cat_model.predict(X_final)[0])])[0]

    with torch.no_grad():
        inp = torch.tensor(X_final.toarray(), dtype=torch.float32).to(device)
        mlp_idx = torch.argmax(mlp_model(inp), dim=1).item()
        mlp_pred = label_enc.inverse_transform([mlp_idx])[0]

    override_label = None
    override_reason = None

    #Post Processing using flags 
    if sem_conf == 2:
        override_label = "REFUTES"

    elif neg_flag == 1:
        e_words = e_clean.split()
        if "not" in e_words:
            idx = e_words.index("not")
            if idx + 1 < len(e_words):
                neg_target = e_words[idx + 1]
                if neg_target in c_clean.split():
                    override_label = "REFUTES"

    elif directneg == 1:
        override_label = "REFUTES"
        override_reason = "Direct negation conflict"

    elif num_contra == 1:
        override_label = "REFUTES"
        override_reason = "Numeric contradiction"

    if override_label is not None:
        return {
            "Logistic Regression": override_label,
            "XGBoost": override_label,
            "LightGBM": override_label,
            "CatBoost": override_label,
            "MLP": override_label
        }

    return {
        "Logistic Regression": lr_pred,
        "XGBoost": xgb_pred,
        "LightGBM": lgb_pred,
        "CatBoost": cat_pred,
        "MLP": mlp_pred
    }

In [8]:
res = predict_claim(
    "Bats are blind",
    "Studies show bats have functional eyes and can see, though they rely heavily on echolocation.",
    "VERIFIABLE"
)
print("\nPredictions:", res)

#Final count
supports = list(res.values()).count("SUPPORTS")
refutes = list(res.values()).count("REFUTES")
nei     = list(res.values()).count("NOT ENOUGH INFO")
if supports > refutes and supports > nei:
    final_label = "SUPPORTS"
elif refutes > supports and refutes > nei:
    final_label = "REFUTES"
else:
    final_label = "NOT ENOUGH INFO"

print("\nFinal Decision:", final_label)


Predictions: {'Logistic Regression': 'REFUTES', 'XGBoost': 'REFUTES', 'LightGBM': 'REFUTES', 'CatBoost': 'REFUTES', 'MLP': 'REFUTES'}

Final Decision: REFUTES
